# How it is asked: general title/abstract task description in the question instructions

**One factor changed relative to [05_fulltext_criteria.ipynb](05_fulltext_criteria.ipynb)** (review information = only the full-text criteria, in `state.review`): a general, review-independent task description is added to the `instructions` of the `include` question, after the fixed sentence. It says this is title/abstract screening, that records are kept when in doubt, and that criteria the abstract cannot show do not count against a record. The text is the same for every review and was written without looking at any labels.

| | `state` | `questions.include.instructions` |
|---|---|---|
| criteria only (comparison) | review: full-text criteria; record: title, abstract | fixed sentence |
| this run | review: full-text criteria; record: title, abstract | fixed sentence + task description |

Labels: CLEF TAR abstract-level (`label_abs`) and content-level (`label_content`).

Nothing paid runs unless `RUN_SMOKE`, `RUN_SUBSET` or `RUN_PAID` is set to `True` below.

In [1]:
import json, sys
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "README.md").exists())
sys.path.insert(0, str(ROOT / "src"))
import screening as S

RUN_SMOKE = False    # one paid call
RUN_SUBSET = False   # the 10 reviews with < 1000 records (3,637 records); see the cost estimate
RUN_PAID = False     # all 20 reviews; read the cost estimate first
OUT = S.out_dir("fulltext_criteria_task_description")
COMPARE = {"baseline": S.out_dir("baseline"),
           "criteria_only": S.out_dir("review_fulltext_criteria_only"),
           "task_description": OUT}

/Users/joshua/Desktop/Repositories/Research/decision-model-screener/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Data

In [2]:
records, CRITERIA = S.load_records()
records = S.add_bm25(records, CRITERIA)
FT = json.loads((ROOT / "data" / "processed" / "tar2019_intervention_fulltext_criteria.json").read_text())
assert set(FT) == set(CRITERIA)
n = records.groupby("review").size()
SUBSET = sorted(n[n < 1000].index)
print("subset:", len(SUBSET), "reviews,", int(n[SUBSET].sum()), "records")

20 reviews, 41996 records, 41183 unique PMIDs
not returned by PubMed: 0 | no abstract: 2183 of 41996


subset: 10 reviews, 3637 records


## Decision model input

In [3]:
TASK = (
    "Task: title and abstract screening for a systematic review. Only the title and abstract of the record are available. "
    "Judge whether the record could meet the criteria; at this stage a record is kept when in doubt. "
    "A criterion that the title and abstract do not report on (for example outcomes, follow-up or risk of bias) does not count against the record. "
    "A record is excluded when it clearly fails a criterion, for example because of the study design, the participants or the intervention."
)

def build_body(review, title, abstract):
    return {
        "model": S.MODEL,
        "state": {
            "review": {"criteria": FT[review]["criteria"]},
            "record": {"title": title, "abstract": abstract},
        },
        "questions": {
            "include": {
                "type": "noul",
                "instructions": S.QUESTIONS["include"]["instructions"] + "\n\n" + TASK,
            }
        },
    }

print(json.dumps(build_body(*records.loc[0, ["review", "title", "abstract"]]), indent=1)[:2500])
print("...")
print(json.dumps(build_body(*records.loc[0, ["review", "title", "abstract"]])["questions"], indent=1))

{
 "model": "typesafe/jev-1.14",
 "state": {
  "review": {
   "criteria": "### Criteria for considering studies for this review\n#### Types of studies\nAll randomised controlled trials (RCTs) using inhaled corticosteroids (ICS) in patients with bronchiectasis, in comparison to placebo.\n#### Types of participants\nChildren or adults with bronchiectasis (defined clinically or radiologically) not related to cystic fibrosis.\n#### Types of interventions\nAll types of ICS.\n#### Types of outcome measures\n##### Primary outcomes\nChange in objective measures of lung function.\n##### Secondary outcomes\n###### (A) for short\u2010term effectiveness (6 months or less)\nMean difference in bronchiectasis severity control (wheeze, dyspnoea, cough diary, etc).\nMean of respiratory exacerbations, or hospitalisations per participant, or both.\nSputum volume.\nMean difference in other objective indices (airway markers of inflammation, exhaled nitric oxide, etc).\nMean difference in quality of life in

## Cost estimate (no API call)

The criteria-only run plus about 470 characters of task description per call. The real cost per call is compared below from the `usage` field.

In [4]:
est = S.estimate_cost(records, build_body)
print(f"subset ({len(SUBSET)} reviews): ${est.loc[SUBSET, 'usd'].sum():.2f}")

TypeSafe | typesafe/jev-1.14-20261014 | USD per 1M input tokens: 0.041999999999999996 | output: 0


requests: 41996 | mean chars per request: 8364 | max: 23386
estimated input tokens: 114.8M | estimated cost: $4.82
subset (10 reviews): $0.29


## Smoke test (one paid call)

Checks that the API accepts long instructions, and compares tokens and cost with the baseline call for the same record.

In [5]:
if RUN_SMOKE:
    rec = records.loc[0, ["review", "pmid", "title", "abstract"]]
    resp = S.call(build_body(rec.review, rec.title, rec.abstract))
    print(json.dumps(resp, indent=1))
    base = {r["pmid"]: r for r in map(json.loads, (COMPARE["criteria_only"] / f"{rec.review}.jsonl").open())}[rec.pmid]["response"]
    print("criteria only, same record:", base["usage"]["input_tokens"], "tokens,", base["usage"]["cost"], "USD, score", base["answers"]["include"]["noul"])

## Subset (paid)

In [6]:
if RUN_SUBSET:
    S.run_reviews(records, SUBSET, build_body, OUT)

## Full run (paid)

In [7]:
if RUN_PAID:
    S.run_reviews(records, CRITERIA, build_body, OUT)

## Results

Both configurations on the same records: the reviews with a cache file for this run. Cost and tokens are summed from the `usage` field of the cached responses for exactly these records.

In [8]:
done = sorted(p.stem for p in OUT.glob("*.jsonl"))
print(len(done), "of", len(CRITERIA), "reviews run")
sub = records[records.review.isin(done)].copy()
for col, path in COMPARE.items():
    sub = S.add_scores(sub, path, col)

results = {c: S.evaluate(sub, c) for c in ("bm25", *COMPARE)}
if done:
    results["task_description - criteria_only"] = results["task_description"] - results["criteria_only"]
    results["task_description - criteria_only"]["reviews"] = results["criteria_only"]["reviews"]
pd.concat(results)

0 of 20 reviews run
baseline: scored 0 of 0 records | model: {'typesafe/jev-1.14-20261014'} | cost: $1.80 | without a score (ranked last): 0


criteria_only: scored 0 of 0 records | model: {'typesafe/jev-1.14-20261014'} | cost: $0.22 | without a score (ranked last): 0
task_description: scored 0 of 0 records | model: set() | cost: $0.00 | without a score (ranked last): 0


reviews
bm25             label_abs            0
                 label_content        0
baseline         label_abs            0
                 label_content        0
criteria_only    label_abs            0
                 label_content        0
task_description label_abs            0
                 label_content        0

In [9]:
def usage(path, reviews):
    tok = cost = n = 0
    for r in reviews:
        for row in map(json.loads, (path / f"{r}.jsonl").open()):
            u = row["response"].get("usage", {})
            tok += u.get("input_tokens", 0); cost += u.get("cost") or 0; n += 1
    return {"calls": n, "input_tokens": tok, "usd": round(cost, 3), "tokens_per_call": round(tok / max(n, 1))}

pd.DataFrame({c: usage(p, done) for c, p in COMPARE.items()}).T

,calls,input_tokens,usd,tokens_per_call
baseline,0,0,0,0
criteria_only,0,0,0,0
task_description,0,0,0,0


In [10]:
# per-review MAP
rows = {}
for r, df in sub.groupby("review"):  # empty until a run has finished
    for label in ("label_abs", "label_content"):
        for c in COMPARE:
            m = S.metrics(df.sort_values([c, "bm25", "pmid"], ascending=[False, False, True])[label])
            rows.setdefault(r, {})[(label, c)] = round(m["MAP"] * 100, 1) if m else None
per_review = pd.DataFrame(rows).T
for label in ("label_abs", "label_content") if rows else ():
    d = per_review[(label, "task_description")] - per_review[(label, "criteria_only")]
    print(f"{label}: task description vs criteria only, better {(d > 0).sum()} / worse {(d < 0).sum()} of {d.notna().sum()} reviews")
per_review

""
